# RAG at a glance

We start with a bare-bones RAG pipeline, from (1) ingesting and chunking a document, to
(2) embeddings and retrieval, to (3) generation.

On Colab, run the cells from the top; the first one fetches the repo (about a minute). On your
own machine, run `uv sync`, then `uv run jupyter lab` from the repo root.

In [ ]:
# On Colab, fetch the repo first (about a minute). On your own machine this is skipped.
import sys
if "google.colab" in sys.modules:
    !git clone -q --depth 1 https://github.com/ecoop/rulebook /content/rulebook
    %pip install -qq -e /content/rulebook
    sys.path.insert(0, "/content/rulebook/src")  # this session can't see the new install otherwise

from rulebook.notebook import setup, show_table
setup()  # work from the repo root and load .env

## 1. Ingest and chunk

Ingesting reads a document into plain text, page by page; `extract_pages` reads PDF, Markdown or
plain text. Chunking then cuts that text into pieces. RAG retrieves *pieces* of documents, not
whole files, and how those pieces are cut is one of the biggest levers on the whole system. Here
the chunker splits each rulebook on its own rule numbers, so each chunk is one rule you can cite.

In [ ]:
from pathlib import Path
from rulebook.ingest import extract_pages
from rulebook.chunking import chunk_pages

doc = Path('rules/checkers/checkers.md')
chunks = chunk_pages(extract_pages(doc), source=doc.name, domain='checkers')

print(f'{len(chunks)} chunks from {doc.name}; the first five:')
show_table(['rule', 'text'], [(c.rule_id, c.text[:80] + '…') for c in chunks[:5]])

The `rule` id on each chunk is what an answer cites, so a reader can check a claim against the
actual rule. Text before the first numbered rule becomes a `preamble` chunk; <a href="https://ecoop.github.io/rulebook/embeddings.html" target="_parent">Embeddings</a> (§4)
shows how that one can get in the way.

## 2. Embed and retrieve

An embedding model computes an **embedding** for each chunk: a vector, a fixed-length list of
numbers, that represents what the chunk says. Chunks that say similar things get vectors that
lie near each other in the model's **embedding space**. The vectors, stored with their chunks,
make up the **index**. Building it runs §1's ingest and chunk on each rulebook, then embeds and
stores each chunk (a few seconds, with a small local model):

In [ ]:
from rulebook.notebook import build_index

build_index('checkers', 'tennis')

To retrieve, the same model computes an embedding for the question, which places it in that same
space, and the chunks nearest to it come back. **Distance** is 2 − 2 × the cosine similarity of
the two vectors: 0 means identical, 2 unrelated, 4 opposite. Lower is nearer.

In [ ]:
from rulebook.retrieve import retrieve

hits = retrieve('can a king move backwards?', domain='checkers', k=3)
show_table(['rank', 'rule', 'distance', 'text'],
           [(i, h.rule_id, f'{h.distance:.2f}', h.text[:80] + '…') for i, h in enumerate(hits, 1)])

Rule 4.2, *How a king moves*, comes first, though the question barely shares its words.

**Go further:** <a href="https://ecoop.github.io/rulebook/embeddings.html" target="_parent">Embeddings</a> looks inside an embedding and shows a question where the
nearest chunk isn't the answer; <a href="https://ecoop.github.io/rulebook/evaluation.html" target="_parent">Evaluation</a> measures retrieval with a set of known
answers instead of by eye.

## 3. Generate

The last stage hands Claude *only* the retrieved chunks, with instructions to cite every claim
as `[domain rule]` and to say so when the chunks don't cover the question. `build_prompt` makes
exactly what Claude is sent; here it is for one question:

In [ ]:
from rulebook.generate import build_prompt

question = 'How does a piece become a king, and what can it then do?'
chunks = retrieve(question, domain='checkers', k=5)
system, user = build_prompt(question, chunks)
print(system, user, sep='\n\n')

Sending that to Claude takes an API key, which the *Generation & citations* notebook (coming)
sets up. So this tour can run without one, here are Claude's answers from a real run, saved with
the repo. The check under each answer runs now: `find_unverified_citations` lists any rule the
answer cites that isn't among the chunks just retrieved.

In [ ]:
import json
from rulebook.generate import find_unverified_citations

saved = json.load(open('notebooks/data/rag_at_a_glance_answers.json'))
answers = {a['question']: a['answer'] for a in saved['answers']}
print(f"Claude ({saved['model']}), recorded {saved['recorded']}:\n")
print(answers[question])
print('\nunverified citations:', find_unverified_citations(answers[question], chunks))

**The guardrail.** Asked about something the rules don't contain, Claude says they don't cover
it, instead of answering from what it already knows.

In [ ]:
print(answers['What is the offside rule in soccer?'])

**Two rulebooks at once.** Retrieval can take the nearest chunks from each of several rulebooks,
and Claude compares them.

In [ ]:
from rulebook.retrieve import retrieve_across_domains

question = 'How do you win each game?'
chunks = retrieve_across_domains(question, ['checkers', 'tennis'], k_per_domain=5)
print('retrieved from:', sorted({c.domain for c in chunks}), '\n')
print(answers[question])
print('\nunverified citations:', find_unverified_citations(answers[question], chunks))

## Where to go next

- **Go further:** <a href="https://ecoop.github.io/rulebook/embeddings.html" target="_parent">Embeddings</a> (<a href="https://colab.research.google.com/github/ecoop/rulebook/blob/main/notebooks/embeddings.ipynb" target="_parent">Colab</a>), then <a href="https://ecoop.github.io/rulebook/evaluation.html" target="_parent">Evaluation</a> (<a href="https://colab.research.google.com/github/ecoop/rulebook/blob/main/notebooks/evaluation.ipynb" target="_parent">Colab</a>), then
  <a href="https://ecoop.github.io/rulebook/reranking.html" target="_parent">Reranking</a> (<a href="https://colab.research.google.com/github/ecoop/rulebook/blob/main/notebooks/reranking.ipynb" target="_parent">Colab</a>), then
  <a href="https://ecoop.github.io/rulebook/chunking.html" target="_parent">Chunking</a> (<a href="https://colab.research.google.com/github/ecoop/rulebook/blob/main/notebooks/chunking.ipynb" target="_parent">Colab</a>), all key-free, and more of these notebooks as they're added (listed on the <a href="https://ecoop.github.io/rulebook/" target="_parent">notebooks page</a>).
- **Turn a knob.** `retrieve` returns the `k` nearest chunks; pass `k=1` or `k=8` in §2 and see
  what comes back. Or edit a rule in `rules/checkers/checkers.md`, rebuild, and retrieve again.
- **Run §3 live.** With an Anthropic API key, `rulebook.pipeline.ask` retrieves and asks Claude in
  one call, about a cent a question; *Generation & citations* (coming) walks through it.
- **Add a real ruleset.** `uv run python -m rulebook.add <slug> <your.pdf>` ingests and indexes a
  rule document; `rules/SOURCES.md` lists recognizable ones.
- **Read the core modules** in order: `ingest` → `chunking` → `embeddings` → `store` → `retrieve` →
  `generate`. The rest of `src/rulebook/` (auth, roles, cost tracking) is app scaffolding.